# DQN Single Episode Visualization

In [ ]:
import os
import sys
sys.path.append('..')
import csv
import torch
from collections import defaultdict
import gymnasium as gym
from src.agent import BusAgent
from src.env import BusEnv

MODEL_PATH = "../model/dqn_agent.pth"
TARGET_STOPS = {0, 2, 7}

Using device: cpu


In [ ]:
def load_agent(env, model_path):
    agent = BusAgent(env)

    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model not found at {model_path}")

    checkpoint = torch.load(model_path, map_location=torch.device("cpu"))

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        agent.main_q.load_state_dict(checkpoint["model_state_dict"])
    else:
        agent.main_q.load_state_dict(checkpoint)

    agent.main_q.eval()
    print(f"Loaded model from {model_path}")

    return agent

In [ ]:
def run_episode(env, agent):
    logs = defaultdict(lambda: defaultdict(lambda: {"arr": [], "dep": []}))
    prev_stops = {}

    # NEW: time-series data
    stop_crowding = {}   # timestep -> list of queue lengths
    bus_occupancy = {}   # timestep -> {bus_id: occupancy}

    state, info = env.reset()
    done = False

    while not done:
        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            q_values = agent.main_q(state_tensor)
        action = torch.argmax(q_values, dim=1).item()

        next_state, reward, terminated, truncated, info = env.step(action)
        timestep = info["timestep"]

        # --- NEW: record stop crowding ---
        stop_crowding[timestep] = env.unwrapped.queues.copy().tolist()

        # --- NEW: record bus occupancy ---
        bus_occupancy[timestep] = {}
        for i, bus in enumerate(env.unwrapped.buses):
            if bus["active"]:
                bus_occupancy[timestep][i] = bus["occupancy"]

        # --- EXISTING: arrival/departure logging ---
        for i, bus in enumerate(env.unwrapped.buses):
            if not bus["active"]:
                continue

            current_stop = bus["stop"]

            if i not in prev_stops:
                prev_stops[i] = current_stop
                continue

            prev_stop = prev_stops[i]

            if current_stop != prev_stop:
                if current_stop in TARGET_STOPS:
                    logs[i][current_stop]["arr"].append(timestep)

                if prev_stop in TARGET_STOPS:
                    logs[i][prev_stop]["dep"].append(timestep)

            prev_stops[i] = current_stop

        state = next_state
        done = terminated or truncated

    return logs, stop_crowding, bus_occupancy

In [ ]:
def save_logs(logs, filename="agent-logs/bus_episode_chart.csv"):
    with open(filename, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["bus_id", "stop", "event", "time"])

        for bus_id, stops in logs.items():
            for stop, data in stops.items():
                for t in data["arr"]:
                    writer.writerow([bus_id, stop, "arrival", t])
                for t in data["dep"]:
                    writer.writerow([bus_id, stop, "departure", t])

    print(f"Saved chart data to {filename}")

In [ ]:
def save_stop_crowding(stop_crowding, filename="agent-logs/stop_crowding.csv"):
    with open(filename, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["timestep", "stop_id", "queue_length"])

        for t, queues in stop_crowding.items():
            for stop_id, length in enumerate(queues):
                writer.writerow([t, stop_id, length])

    print(f"Saved stop crowding data to {filename}")


def save_bus_occupancy(bus_occupancy, filename="agent-logs/bus_occupancy.csv"):
    with open(filename, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["timestep", "bus_id", "occupancy"])

        for t, buses in bus_occupancy.items():
            for bus_id, occ in buses.items():
                writer.writerow([t, bus_id, occ])

    print(f"Saved bus occupancy data to {filename}")

In [ ]:
# Register environment
gym.register(
    id="gymnasium_env/BusRouting-v0",
    entry_point=BusEnv,
)

env = gym.make("gymnasium_env/BusRouting-v0")

# Load trained agent
agent = load_agent(env, MODEL_PATH)

# Run one episode
logs, stop_crowding, bus_occupancy = run_episode(env, agent)

env.close()

# Save results
save_logs(logs)
save_stop_crowding(stop_crowding)
save_bus_occupancy(bus_occupancy)

/tmp/ipykernel_162981/4024214799.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=torch.device("cpu"))


RuntimeError: [enforce fail at inline_container.cc:174] . file in archive is not in a subdirectory: data